# Type Hints in Python

> **Interview Note:** Type hints are now standard in modern Python (3.9+). AI/ML codebases (PyTorch, Transformers, LangChain, FastAPI) rely heavily on them. Understanding `typing` module is essential.

---

## 1. Basics (Python 3.5+)

In [ ]:
# Function annotations
def greet(name: str) -> str:
    return f"Hello, {name}"

# Variable annotations
count: int = 0
names: list[str] = []  # Python 3.9+
scores: dict[str, float] = {}

# Class attributes
class Config:
    learning_rate: float = 0.001
    batch_size: int = 32
    device: str = "cuda"

### Python Version Differences

| Feature | 3.9+ | 3.10+ | 3.11+ |
|---------|------|-------|-------|
| Built-in generics | `list[str]` | ✓ | ✓ |
| Union syntax | `list[str] \| None` | `X \| Y` | ✓ |
| `Self` type | `from typing_extensions import Self` | ✓ | ✓ |
| `TypeAliasType` | ✗ | ✗ | ✓ |
| `assert_never` | ✗ | ✗ | ✓ |

> **Recommendation:** Use `from __future__ import annotations` (3.7+) for postponed evaluation — allows forward references and cleaner syntax.

---

## 2. Core `typing` Types

In [ ]:
from typing import (
    List, Dict, Set, Tuple, Optional, Union,
    Sequence, Mapping, Iterable, Iterator,
    Callable, Any, Literal, Final, ClassVar,
    Type, TypeVar, Generic, Protocol,
    overload, cast, get_type_hints
)

# Collections (legacy - use built-ins in 3.9+)
legacy_list: List[int] = [1, 2, 3]
legacy_dict: Dict[str, float] = {"a": 1.0}
legacy_tuple: Tuple[int, str, bool] = (1, "hi", True)
legacy_set: Set[str] = {"a", "b"}

# Modern (3.9+)
modern_list: list[int] = [1, 2, 3]
modern_dict: dict[str, float] = {"a": 1.0}
modern_tuple: tuple[int, str, bool] = (1, "hi", True)
modern_set: set[str] = {"a", "b"}

### Special Types

In [ ]:
# Optional = Union[T, None]
def find_user(user_id: int) -> Optional[dict]:
    if user_id < 0:
        return None
    return {"id": user_id}

# Union (multiple possible types)
def process(input: Union[str, int, float]) -> str:
    return str(input)

# Modern union syntax (3.10+)
def process_modern(input: str | int | float) -> str:
    return str(input)

# Literal - specific values
from typing import Literal
Mode = Literal["train", "eval", "inference"]

def set_mode(mode: Mode) -> None:
    print(f"Mode: {mode}")

set_mode("train")
# set_mode("invalid")  # Type checker error

# Final - constants
from typing import Final
MAX_EPOCHS: Final[int] = 100
# MAX_EPOCHS = 200  # Type checker error

# ClassVar - class-level only
from typing import ClassVar
class ModelConfig:
    default_lr: ClassVar[float] = 0.001
    lr: float = 0.01  # instance variable

---

## 3. Callable & Function Types

In [ ]:
from typing import Callable, ParamSpec, Concatenate

# Callable[[ArgTypes], ReturnType]
Transformer = Callable[[str], str]

def apply_transform(text: str, fn: Transformer) -> str:
    return fn(text)

apply_transform("hello", str.upper)
apply_transform("hello", lambda x: x + "!")

# ParamSpec for decorators (3.10+)
P = ParamSpec("P")

def log_calls(fn: Callable[P, int]) -> Callable[P, int]:
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> int:
        print(f"Calling {fn.__name__}")
        return fn(*args, **kwargs)
    return wrapper

@log_calls
def add(a: int, b: int) -> int:
    return a + b

add(1, 2)

---

## 4. Generics & TypeVar

In [ ]:
from typing import TypeVar, Generic

T = TypeVar("T")

# Generic class
class Response(Generic[T]):
    def __init__(self, data: T, success: bool = True):
        self.data = data
        self.success = success

# Type inferred at use site
resp_int = Response(42)
resp_str = Response("hello")
resp_list = Response([1, 2, 3])

# Bounded TypeVar
Number = TypeVar("Number", int, float)

def add_numbers(a: Number, b: Number) -> Number:
    return a + b

add_numbers(1, 2)      # int
add_numbers(1.5, 2.5)  # float
# add_numbers("a", "b")  # Error: str not in bound

# Covariant/Contravariant (advanced)
from typing import TypeVar
T_co = TypeVar("T_co", covariant=True)
T_contra = TypeVar("T_contra", contravariant=True)

---

## 5. Protocols (Structural Subtyping / Duck Typing)

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class SupportsTrain(Protocol):
    def train(self) -> None: ...
    def eval(self) -> None: ...
    @property
    def training(self) -> bool: ...

# Any class with these methods qualifies (no inheritance needed)
class MyModel:
    def __init__(self):
        self._training = True
    def train(self) -> None:
        self._training = True
    def eval(self) -> None:
        self._training = False
    @property
    def training(self) -> bool:
        return self._training

def run_epoch(model: SupportsTrain) -> None:
    model.train()
    # ... training step
    model.eval()

run_epoch(MyModel())  # Works! No inheritance from SupportsTrain

# isinstance works with @runtime_checkable
print(isinstance(MyModel(), SupportsTrain))

> **Why Protocols Matter for AI:** Libraries like PyTorch (`nn.Module`), Transformers (`PreTrainedModel`), and LangChain (`Runnable`) use protocols. Your code can accept any object with the right interface without coupling to specific classes.

---

## 6. Overloads (Multiple Signatures)

In [ ]:
from typing import overload

class Tensor:
    @overload
    def __getitem__(self, key: int) -> float: ...
    
    @overload
    def __getitem__(self, key: slice) -> "Tensor": ...
    
    @overload
    def __getitem__(self, key: tuple) -> "Tensor": ...
    
    def __getitem__(self, key):
        # Implementation
        pass

# Type checker knows:
t = Tensor()
x: float = t[0]        # float
y: Tensor = t[1:5]     # Tensor
z: Tensor = t[0, 1:3]  # Tensor

---

## 7. `Self` Type (Python 3.11+)

In [ ]:
# For fluent interfaces / builder patterns
try:
    from typing import Self
except ImportError:
    from typing_extensions import Self

class ModelBuilder:
    def __init__(self):
        self._layers = []
    
    def add_layer(self, layer) -> Self:
        self._layers.append(layer)
        return self
    
    def set_optimizer(self, opt) -> Self:
        self._optimizer = opt
        return self
    
    def build(self) -> "Model":
        return Model(self._layers, self._optimizer)

class Model:
    def __init__(self, layers, optimizer):
        self.layers = layers
        self.optimizer = optimizer

# Chaining works with correct return type
model = ModelBuilder().add_layer("L1").add_layer("L2").set_optimizer("adam").build()

---

## 8. TypedDict (Structured Dicts)

In [ ]:
from typing import TypedDict, NotRequired, Required

class TrainConfig(TypedDict):
    epochs: int
    lr: float
    batch_size: NotRequired[int]  # Optional key
    device: Required[str]  # Explicit required (3.11+)

config: TrainConfig = {
    "epochs": 10,
    "lr": 0.001,
    "device": "cuda"
    # batch_size optional
}

# Total/Partial
from typing import TypedDict

class PartialConfig(TypedDict, total=False):
    epochs: int
    lr: float

partial: PartialConfig = {}  # All keys optional

---

## 9. Pydantic (Runtime Validation + Types)

In [ ]:
# pip install pydantic
try:
    from pydantic import BaseModel, Field, validator, computed_field
    from pydantic import ConfigDict
    
    class ModelConfig(BaseModel):
        model_config = ConfigDict(extra='forbid', frozen=True)
        
        name: str
        epochs: int = Field(ge=1, le=1000)
        lr: float = Field(gt=0, lt=1)
        batch_size: int = 32
        device: str = "cuda"
        
        @computed_field
        @property
        def total_steps(self) -> int:
            return self.epochs * 1000 // self.batch_size
        
        @validator('lr')
        def lr_not_too_small(cls, v):
            if v < 1e-6:
                raise ValueError('lr too small')
            return v

    # Validation at runtime
    cfg = ModelConfig(name="bert", epochs=10, lr=0.001)
    print(cfg)
    print(f"Total steps: {cfg.total_steps}")
    
    # Serialization
    print(cfg.model_dump())
    print(cfg.model_dump_json())
    
    # Validation error
    try:
        ModelConfig(name="test", epochs=0, lr=0.001)
    except Exception as e:
        print(f"Validation error: {e}")

except ImportError:
    print("pydantic not installed. Run: pip install pydantic")

> **AI Use Case:** Pydantic is the standard for config validation, API schemas (FastAPI), LLM function calling (OpenAI function calling uses JSON Schema → Pydantic), and structured output parsing.

---

## 10. Type Introspection

In [ ]:
from typing import get_type_hints, get_origin, get_args

def func(a: int, b: list[str]) -> dict[str, float]:
    return {}

# Get all hints
hints = get_type_hints(func)
print(hints)

# Deconstruct generic types
from typing import List
origin = get_origin(List[int])
args = get_args(List[int])
print(f"Origin: {origin}, Args: {args}")

# Works with Union, Optional, Tuple, Callable, etc.
print(get_origin(Union[int, str]))
print(get_args(Union[int, str]))

# Annotated (3.9+) - metadata attached to types
from typing import Annotated
Id = Annotated[int, "user id", "positive"]
print(get_origin(Id))
print(get_args(Id))

---

## 11. Common Patterns in AI Codebases

In [ ]:
# 1. Config classes with defaults
from dataclasses import dataclass
from typing import Optional

@dataclass
class TrainingConfig:
    model_name: str
    learning_rate: float = 3e-4
    batch_size: int = 32
    max_seq_len: int = 512
    warmup_steps: int = 100
    weight_decay: float = 0.01
    gradient_accumulation: int = 1
    fp16: bool = False
    output_dir: str = "./outputs"
    
    def __post_init__(self):
        assert self.learning_rate > 0

# 2. Dataset item protocol
from typing import Protocol

class Batch(Protocol):
    input_ids: "torch.Tensor"
    attention_mask: "torch.Tensor"
    labels: Optional["torch.Tensor"]

# 3. Model forward signature
from typing import Union

def forward(
    self,
    input_ids: "torch.Tensor",
    attention_mask: Optional["torch.Tensor"] = None,
    labels: Optional["torch.Tensor"] = None,
    **kwargs
) -> Union[Tuple["torch.Tensor"], "ModelOutput"]:
    ...

# 4. Collate function
from typing import List

CollateFn = Callable[[List[dict]], dict]

# 5. Optimizer param groups
ParamGroup = dict[str, Union[float, List["torch.nn.Parameter"]]]
OptimizerConfig = List[ParamGroup]

---

## 12. Type Checking Tools

In [ ]:
# mypy - static type checker
# pip install mypy
# mypy your_file.py

# pyright - Microsoft's fast type checker (VS Code Pylance)
# pip install pyright

# ruff - fast linter with type checking (recommended)
# pip install ruff
# ruff check .

# py.typed marker - tells tools your package supports typing
# Create empty file: your_package/py.typed

---

## 13. Interview Questions

1. **Difference between `List[int]` and `list[int]`?**
   - `list[int]` is built-in generic (3.9+), `List[int]` is from `typing` (legacy)

2. **What is `Optional[T]` equivalent to?**
   - `Union[T, None]`

3. **When to use `TypeVar` vs `Any`?**
   - `TypeVar`: Generic code that preserves type relationships
   - `Any`: Opt-out of type checking (use sparingly)

4. **What's the difference between `Protocol` and `ABC`?**
   - `Protocol`: Structural subtyping (duck typing), no inheritance needed
   - `ABC`: Nominal subtyping, requires explicit inheritance

5. **What does `@overload` do?**
   - Provides multiple type signatures for one implementation; only the last (non-`@overload`) is executed

6. **What is `Self` type used for?**
   - Fluent interfaces, builder patterns, cloning — return type is "same as subclass"

7. **`TypedDict` vs `dataclass` vs `Pydantic BaseModel`?**
   - `TypedDict`: Dict shape only, no runtime validation
   - `dataclass`: Class with auto-generated `__init__`, minimal runtime overhead
   - `Pydantic`: Full validation, serialization, coercion, defaults

8. **What is variance (covariant/contravariant)?**
   - Covariant: `List[Dog]` is subtype of `List[Animal]` (read-only)
   - Contravariant: `Callable[[Animal], ...]` is subtype of `Callable[[Dog], ...]` (write-only)

9. **How does `get_type_hints()` differ from `__annotations__`?**
   - `get_type_hints()`: Resolves forward references, evaluates strings, handles `__future__.annotations`
   - `__annotations__`: Raw dict, may contain strings

10. **Why use `from __future__ import annotations`?**
    - Postpones evaluation (annotations stored as strings), allows forward refs, faster import, cleaner syntax